In [1]:
# Let's add the root directory to our system search path to allow imports from sibling directories.
import os, sys
module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
import gc, random
from pathlib import Path
from typing import List

import cv2
import torch
import numpy as np
from torchvision.transforms import v2

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Import your modules
from src.dino import ConvNeXtV2
from src.dataset import SonarDataset
from src.utils import show_images, load_backbone, run_inference

WEIGHTS_DIR = Path("../weights/")
EPOCH_COUNT = len(list(WEIGHTS_DIR.iterdir())) - 2

DEVICE = torch.device("cuda")  # ConvNeXtV2 doesn't support CPU

SEED = 42
RNG = np.random.default_rng(seed=SEED)

DATASET_PATH = Path("../data/labelled/s3seg")
IMGS_PATH = DATASET_PATH / "images/"
MASK_PATH = DATASET_PATH / "masks/"
EMBEDS_PATH = DATASET_PATH / "embeds/"

DATA_EXT = '.tiff'
DATA_FILES = sorted([im.stem for im in IMGS_PATH.glob(f"*{DATA_EXT}")])

N = len(DATA_FILES)
TEST_SPLIT = int(0.2 * N)

/home/hayat/projects/benthic/venv/lib/python3.8/site-packages/spconv/pytorch/functional.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  _TORCH_CUSTOM_FWD = amp.custom_fwd(cast_inputs=torch.float16)
/home/hayat/projects/benthic/venv/lib/python3.8/site-packages/spconv/pytorch/functional.py:97: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/home/hayat/projects/benthic/venv/lib/python3.8/site-packages/spconv/pytorch/functional.py:163: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/home/hayat/projects/benthic/venv/lib/python3.8/site-packages/spconv/pytorch/functional.py:243: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Pl

In [3]:
EMBEDS_PATH.mkdir(exist_ok=True)

embed_files = list(EMBEDS_PATH.glob('*.npz'))
if len(embed_files) < N:
    model = load_backbone(WEIGHTS_DIR / "checkpoint_latest.pth")

    for file in DATA_FILES:
        embed_path = EMBEDS_PATH / (file + '.npz')
        if embed_path.exists():
            continue

        img_path = IMGS_PATH / (file + DATA_EXT)
        img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED).astype(np.float32)
        img = torch.from_numpy(img[np.newaxis, :, :])  # (W, H) -> (1, W, H)

        cls, flat_patches, outputs = run_inference(model, img)
        np.savez_compressed(
            embed_path,
            cls=cls,
            stage0=outputs[0],
            stage1=outputs[1],
            stage2=outputs[2],
            stage3=outputs[3],
            stage4=outputs[4],  # Fused Hypercolumn
        )

In [ ]:
masks, embeds = [], []
for file in DATA_FILES:
        mask_path = MASK_PATH / (file + DATA_EXT)
        embed_path = EMBEDS_PATH / (file + '.npz')

        mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED).astype(np.float32)
        masks.append(mask)

        data = np.load(embed_path)
        embeds.append((
            data['cls'],
            data['stage0'],
            data['stage1'],
            data['stage2'],
            data['stage3'],
            data['stage4'],
        ))

In [ ]:
shuffled_indices = RNG.permutation(N)
test_indices = shuffled_indices[:TEST_SPLIT]
train_indices = shuffled_indices[TEST_SPLIT:]

# Few Shot Scalability (1%, 5%, 10%, 100%)
few_shot = (0.01, 0.05, 0.10, 1.00)

# Save the trained classifier for each stage for each scalability limit
knn_probes = [[] for _ in range(5)]

In [ ]:
probes = [[] for _ in range(5)]
for stage in range(5):
    X_train = np.array([embeds[i][stage + 1] for i in train_indices])
    y_raw = np.array([masks[i] for i in train_indices])

    N_train, H_feat, W_feat, C = X_train.shape
    _, H_orig, W_orig = y_raw.shape

    # Exact Nearest-Neighbor mask downsampling (center-aligned)
    row_idx = ((np.arange(H_feat) + 0.5) * (H_orig / H_feat)).astype(int)
    col_idx = ((np.arange(W_feat) + 0.5) * (W_orig / W_feat)).astype(int)
    y_train = y_raw[:, row_idx[:, None], col_idx]

    for shot in few_shot:
        num_samples = int(shot * (N - TEST_SPLIT))

        X_samples = X_train[:num_samples].reshape(-1, X_train.shape[-1])  # (N * H * W, C)
        y_samples = y_train[:num_samples].reshape(-1)  # (N * H * W)

        linear_probe = make_pipeline(
            StandardScaler(),
            SGDClassifier(
                loss="log_loss",          # Logistic regression loss
                penalty="l2",             # L2 (Ridge) regularization
                alpha=1e-4,               # Regularization strength (smaller = less regularized)
                max_iter=1000,
                tol=1e-3,
                early_stopping=True,      # Halts early if validation score plateaus
                validation_fraction=0.1,  # 10% held out for early stopping
                class_weight="balanced",  # balances background vs foreground pixels
                random_state=SEED,
            )
        )
        linear_probe.fit(X_samples, y_samples)

        knn_probe = make_pipeline(
            StandardScaler(),
            KNeighborsClassifier(
                n_neighbors=10,      # Based on Two-NN test
                weights="distance",  # Weights closer neighbors higher
                algorithm="auto",    # Automatically chooses kd_tree/ball_tree/brute
                n_jobs=-1            # Use all CPU cores for parallel queries
            )
        )
        knn_probe.fit(X_samples, y_samples)

        probes[stage].append((linear_probe, knn_probe))
